In [1]:
import pandas as pd
import numpy as np
import re

---

In [2]:
df = pd.read_csv("../../../datasets/raw/steam_games_requirements.csv")
df2 = pd.read_csv("../../../datasets/scraped/steam_requirements_scraped.csv")

---

In [3]:
df.head(1)

,Unnamed: 0,url,types,name,desc_snippet,recent_reviews,all_reviews,release_date,developer,publisher,...,game_details,languages,achievements,genre,game_description,mature_content,minimum_requirements,recommended_requirements,original_price,discount_price
0,0,https://store.steampowered.com/app/379720/DOOM/,app,DOOM,Now includes all three premium DLC packs (Unto...,"Very Positive,(554),- 89% of the 554 user revi...","Very Positive,(42,550),- 92% of the 42,550 use...","May 12, 2016",id Software,"Bethesda Softworks,Bethesda Softworks",...,"Single-player,Multi-player,Co-op,Steam Achieve...","English,French,Italian,German,Spanish - Spain,...",54.0,Action,"About This Game Developed by id software, the...",NaN,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",$19.99,$14.99


In [4]:
df = df[df['types'] == 'app']
df.drop(columns=['types'], inplace=True)

In [5]:
df['url'] = df['url'].apply(lambda row: row.split('/')[4])
df.rename(columns={'url': 'app_id'}, inplace=True)

In [6]:
df = df[['app_id', 'name', 'minimum_requirements', 'recommended_requirements']]

In [7]:
df.head(1)

,app_id,name,minimum_requirements,recommended_requirements
0,379720,DOOM,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers..."


In [8]:
df['app_id'] = df['app_id'].astype('int64')

In [9]:
df.isna().sum()

app_id                          0
name                           14
minimum_requirements        16952
recommended_requirements    16946
dtype: int64

In [10]:
df.dropna(how='all', inplace=True)

In [11]:
df = df[~df['name'].str.contains(r'\b(OST|Soundtrack)\b')]

C:\Users\wastedy\AppData\Local\Temp\ipykernel_12136\3679353767.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df['name'].str.contains(r'\b(OST|Soundtrack)\b')]


In [12]:
df[df[['minimum_requirements', 'recommended_requirements']].isna().all(axis=1)] # Missing

,app_id,name,minimum_requirements,recommended_requirements
16,597170,Clone Drone in the Danger Zone,NaN,NaN
20,42700,Call of Duty®: Black Ops,NaN,NaN
24,12210,Grand Theft Auto IV,NaN,NaN
26,400,Portal,NaN,NaN
28,704450,Neverwinter Nights: Enhanced Edition,NaN,NaN
...,...,...,...,...
40813,912210,Achievement Collector: Cat,NaN,NaN
40815,912140,SpaceBall in Cube,NaN,NaN
40824,906470,Gravia,NaN,NaN
40826,906430,Alive,NaN,NaN


---

In [13]:
# merge with scraped dataset
df2

,steam_appid,name,pc_requirements_minimum,pc_requirements_recommended
0,3065800,Marathon,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
1,2483190,Forza Horizon 6,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
2,1962700,Subnautica 2,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
3,1808500,ARC Raiders,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
4,2807960,Battlefield™ 6,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
...,...,...,...,...
3647,4332910,Blindshot Arena,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
3648,3699390,Little Mage,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
3649,1423820,Miner's Hell,"<strong>Minimum:</strong><br><ul class=""bb_ul""...","<strong>Recommended:</strong><br><ul class=""bb..."
3650,4483420,Luna Has Gone,"<strong>Minimum:</strong><br><ul class=""bb_ul""...",NaN


In [14]:
def clean_tags(text):
    if pd.isna(text):
        return None
    if '<br>' in text:
        text = re.sub('<br>', ',', text)
    if '\n' in text:
        text = re.sub('\n', ',', text)
    return " ".join(re.sub(r'<.*?>', ' ', text).split())

In [15]:
df2.dtypes

steam_appid                    int64
name                             str
pc_requirements_minimum          str
pc_requirements_recommended      str
dtype: object

In [16]:
df2.isna().sum()

steam_appid                      0
name                             0
pc_requirements_minimum          6
pc_requirements_recommended    903
dtype: int64

In [17]:
df2.dropna(subset=['name', 'pc_requirements_minimum'], inplace=True) # The games here without names are irrelevant DLCs, games without minimum requirements are OSTs.

In [18]:
df2 = df2[~df2['name'].str.contains(r'\b(OST|Soundtrack)\b')] # Cleaning OSTs by string name

C:\Users\wastedy\AppData\Local\Temp\ipykernel_12136\3624869109.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df2 = df2[~df2['name'].str.contains(r'\b(OST|Soundtrack)\b')] # Cleaning OSTs by string name


In [19]:
df2.isna().sum()

steam_appid                      0
name                             0
pc_requirements_minimum          0
pc_requirements_recommended    899
dtype: int64

In [20]:
df2['min_req'] = df2['pc_requirements_minimum'].apply(lambda x: clean_tags(x))

In [21]:
df2['rec_req'] = df2['pc_requirements_recommended'].apply(lambda x: clean_tags(x))

In [22]:
df2['min_req'][0]

'Minimum: , Requires a 64-bit processor and operating system, OS: Windows 10 64-bit (latest Service Pack), Processor: Intel Core i5-6600 / AMD Ryzen 5 2600, Memory: 8 GB RAM, Graphics: NVIDIA GeForce GTX 1050 Ti (4 GB) / AMD Radeon RX 5500 XT (4 GB) / Intel Arc A580 (8 GB, with ReBAR on), DirectX: Version 12, Network: Broadband Internet connection'

In [23]:
df2.drop(columns=['pc_requirements_minimum', 'pc_requirements_recommended'], inplace=True)

In [24]:
df2['new'] = df2['steam_appid'].apply(lambda x: 'False' if(x) in df['app_id'] else 'True')

In [25]:
df2

,steam_appid,name,min_req,rec_req,new
0,3065800,Marathon,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and...",True
1,2483190,Forza Horizon 6,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and...",True
2,1962700,Subnautica 2,"Minimum: , OS: Windows 10/11, Processor: Intel...","Recommended: , OS: Windows 11, Processor: Inte...",True
3,1808500,ARC Raiders,"Minimum: , OS: Windows 10 or later 64-bit (lat...","Recommended: , OS: Windows 10 or later 64-bit ...",True
4,2807960,Battlefield™ 6,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and...",True
...,...,...,...,...,...
3647,4332910,Blindshot Arena,"Minimum: , OS: Windows 10, Processor: Intel Co...","Recommended: , OS: Windows 10, Processor: Inte...",True
3648,3699390,Little Mage,"Minimum: , OS *: Windows 7 (64bit)., Processor...","Recommended: , OS: Windows 10 (64bit)., Proces...",True
3649,1423820,Miner's Hell,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and...",True
3650,4483420,Luna Has Gone,"Minimum: , OS *: Windows® 7/8/8.1/10/11, Proce...",NaN,True


In [26]:
df3 = df2[df2['new'] == 'True'] # Only new games
df2.drop(columns=['new'], inplace=True)
df3.columns = ['app_id', 'name_x', 'minimum_requirements', 'recommended_requirements', 'new']

In [27]:
df3.info()

<class 'pandas.DataFrame'>
Index: 3645 entries, 0 to 3651
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   app_id                    3645 non-null   int64
 1   name_x                    3645 non-null   str  
 2   minimum_requirements      3645 non-null   str  
 3   recommended_requirements  2746 non-null   str  
 4   new                       3645 non-null   str  
dtypes: int64(1), str(4)
memory usage: 170.9 KB


In [28]:
df3.drop(df3[df3['minimum_requirements'].isna() & df3['recommended_requirements'].isna()].index, inplace=True) # Dropping where both requirements columns are NaN

In [29]:
merged = df.merge(df2, left_on='app_id', right_on='steam_appid', how='left', indicator=True)

In [30]:
merged

,app_id,name_x,minimum_requirements,recommended_requirements,steam_appid,name_y,min_req,rec_req,_merge
0,379720,DOOM,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",NaN,NaN,NaN,NaN,left_only
1,578080,PLAYERUNKNOWN'S BATTLEGROUNDS,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,NaN,NaN,NaN,left_only
2,637090,BATTLETECH,"Minimum:,Requires a 64-bit processor and opera...","Recommended:,Requires a 64-bit processor and o...",NaN,NaN,NaN,NaN,left_only
3,221100,DayZ,"Minimum:,OS:,Windows 7/8.1 64-bit,Processor:,I...","Recommended:,OS:,Windows 10 64-bit,Processor:,...",NaN,NaN,NaN,NaN,left_only
4,8500,EVE Online,"Minimum:,OS:,Windows 7,Processor:,Intel Dual C...","Recommended:,OS:,Windows 10,Processor:,Intel i...",NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...
36117,899836,Rocksmith® 2014 Edition – Remastered – Sabaton...,"Minimum:,OS:,Windows Vista, Windows 7, Windows...","Recommended:,OS:,Windows Vista, Windows 7, Win...",NaN,NaN,NaN,NaN,left_only
36118,899832,Rocksmith® 2014 Edition – Remastered – Stone T...,"Minimum:,OS:,Windows Vista, Windows 7, Windows...","Recommended:,OS:,Windows Vista, Windows 7, Win...",NaN,NaN,NaN,NaN,left_only
36119,906840,Fantasy Grounds - Quests of Doom 4: A Midnight...,"Minimum:,OS:,Windows 7x , 8x or 10x,Processor:...","Recommended:,OS:,Windows 7x , 8x or 10x,Proces...",NaN,NaN,NaN,NaN,left_only
36120,906635,Mega Man X5 Sound Collection,"Minimum:,OS:,WINDOWS® 7 (64bit),Processor:,Int...","Recommended:,OS:,WINDOWS®10 (64bit),Processor:...",NaN,NaN,NaN,NaN,left_only


In [31]:
merged.drop(merged[merged['name_x'].isna() & merged['name_y'].isna()].index, inplace=True) # Dropping where both name columns are NaN

In [32]:
merged.drop(merged[merged['name_x'].isna()].index, inplace=True)

In [33]:
merged.drop(columns=['steam_appid', 'name_y'], inplace=True)

In [34]:
merged = pd.concat([merged, df3]) # New games

In [35]:
merged[merged['app_id'] == 914210]

,app_id,name_x,minimum_requirements,recommended_requirements,min_req,rec_req,_merge,new
34741,914210,N1RV Ann-A: Cyberpunk Bartender Action,NaN,NaN,"Minimum: , OS *: Windows 7, Processor: Intel C...",NaN,both,NaN
209,914210,N1RV Ann-A: Cyberpunk Bartender Action,"Minimum: , OS *: Windows 7, Processor: Intel C...",NaN,NaN,NaN,NaN,True


In [36]:
merged[merged['app_id'].duplicated()]

,app_id,name_x,minimum_requirements,recommended_requirements,min_req,rec_req,_merge,new
15044,200260,Batman: Arkham City - Game of the Year Edition,NaN,NaN,NaN,NaN,left_only,NaN
7,2483190,Forza Horizon 6,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and...",NaN,NaN,NaN,True
209,914210,N1RV Ann-A: Cyberpunk Bartender Action,"Minimum: , OS *: Windows 7, Processor: Intel C...",NaN,NaN,NaN,NaN,True
211,612400,The Last Night,"Minimum: , OS: Windows 10","Recommended: , OS: Windows 10",NaN,NaN,NaN,True
308,1051440,Ship Builder,"Minimum: , OS *: Windows 7/8/8.1/10/11, Proces...",NaN,NaN,NaN,NaN,True
...,...,...,...,...,...,...,...,...
3405,1003900,CHROMATOSE,"Minimum: , OS *: Ideally Windows 8 or later, P...",NaN,NaN,NaN,NaN,True
3424,259000,Dead Pixels II,"Minimum: , OS *: Windows XP., Processor: 2.4gh...","Recommended: , OS *: Windows 7 onwards, Proces...",NaN,NaN,NaN,True
3501,990970,Be the Ruler: Britannia,"Minimum: , OS: Windows 10 or later, Processor:...",NaN,NaN,NaN,NaN,True
3503,490360,Hermodr,"Minimum: , OS *: Winows 7, 8, 10, Processor: C...",NaN,NaN,NaN,NaN,True


In [37]:
merged['minimum_requirements'] = merged['minimum_requirements'].fillna(merged['min_req'])

In [38]:
merged['recommended_requirements'] = merged['recommended_requirements'].fillna(merged['rec_req'])

In [39]:
merged.dropna(subset=['minimum_requirements', 'recommended_requirements', 'min_req', 'rec_req'], how='all', inplace=True)

In [40]:
merged.drop(columns=['min_req', 'rec_req', '_merge', 'new'], inplace=True)

In [41]:
merged.columns = ['appid', 'name', 'min_req', 'rec_req']

In [42]:
merged[merged['appid'] == 888780].values

array([[888780, 'Adapt',
        'Minimum:,OS:,Windows 10,Processor:,2.4 GHz Dual Core or equivalent,Memory:,1 GB RAM,Graphics:,Nvidia GeForce GTX 650 or equivalent,DirectX:,Version 10,Storage:,1 GB available space',
        'Recommended:,OS:,Windows 10,Processor:,3.2 GHz i5 Dual Core or equivalent,Memory:,2 GB RAM,Graphics:,Nvidia GeForce GTX 960 or equivalent,DirectX:,Version 10,Storage:,1 GB available space'],
       [888780, 'Adapt',
        'Minimum: , Requires a 64-bit processor and operating system, OS: Windows 10 (or later), Processor: 3.0 GHz Dual Core or equivalent, Memory: 4 GB RAM, Graphics: NVIDIA GeForce GTX 1050 / AMD Radeon RX 560 (or Greater), DirectX: Version 10, Storage: 2 GB available space',
        'Recommended: , Requires a 64-bit processor and operating system, OS: Windows 10 (or later), Processor: 3.2 GHz Quad Core or equivalent, Memory: 8 GB RAM, Graphics: NVIDIA GeForce GTX 1660 TI / AMD Radeon RX 6500 XT (or Greater), DirectX: Version 10, Storage: 2 GB avail

In [43]:
merged.drop_duplicates(subset=['appid'], keep='last', inplace=True, ignore_index=True)

In [44]:
merged.drop_duplicates(inplace=True, ignore_index=True)

In [45]:
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 24123 entries, 0 to 24122
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   appid    24123 non-null  int64
 1   name     24123 non-null  str  
 2   min_req  24117 non-null  str  
 3   rec_req  23224 non-null  str  
dtypes: int64(1), str(3)
memory usage: 754.0 KB


In [51]:
merged = merged.sort_values(by='appid')
merged.reset_index(drop=True, inplace=True)
merged

,appid,name,min_req,rec_req
0,440,Team Fortress 2,"Minimum:,OS:,Windows® 7 (32/64-bit)/Vista/XP,P...","Recommended:,OS:,Windows® 7 (32/64-bit),Proces..."
1,500,Left 4 Dead,"Minimum:,Supported OS:,Windows® 7 32/64-bit / ...","Recommended:,Supported OS:,Windows® 7 32/64-bi..."
2,550,Left 4 Dead 2,"Minimum:,OS:,Windows® 7 32/64-bit / Vista 32/6...","Recommended:,OS:,Windows® 7 32/64-bit / Vista ..."
3,630,Alien Swarm,"Minimum:,OS:,Windows® 7 / Vista / Vista64 / XP...","Recommended:,OS:,Windows® 7 / Vista / Vista64 ..."
4,1230,Mare Nostrum,"Minimum:,OS: Ubuntu 12.04 LTS, fully updated,P...","Recommended:,OS: Ubuntu 12.04 LTS, fully updat..."
...,...,...,...,...
24118,4656700,Papa's Mocharia Deluxe,"Minimum: , OS *: Windows 7 or newer, Processor...",NaN
24119,4683260,Project Rabbit,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and..."
24120,4702700,Heathen,"Minimum: , Requires a 64-bit processor and ope...","Recommended: , Requires a 64-bit processor and..."
24121,4704000,Shark Mart,"Minimum: , OS: Windows 10, Processor: AMD Ryze...","Recommended: , OS: Windows 11, Processor: AMD ..."


In [49]:
#merged.to_csv('../../../datasets/processed/mergedDF.csv') # uncomment for generating the file